In [0]:
ball_by_ball_df = spark.read.csv("/Volumes/workspace/karan/ipl/Ball_By_Ball.csv/", header=True, inferSchema=True)
match_df = spark.read.csv("/Volumes/workspace/karan/ipl/Match.csv", header=True, inferSchema=True)
player_df = spark.read.csv("/Volumes/workspace/karan/ipl/Player.csv/", header=True, inferSchema=True)
team_df = spark.read.csv("/Volumes/workspace/karan/ipl/Team.csv", header=True, inferSchema=True)
player_match_df = spark.read.csv("/Volumes/workspace/karan/ipl/Player_match.csv", header=True, inferSchema=True)

ball_by_ball_df.createOrReplaceTempView("Ball_By_Ball")
match_df.createOrReplaceTempView("Match")
player_df.createOrReplaceTempView("Player")
team_df.createOrReplaceTempView("Team")
player_match_df.createOrReplaceTempView("Player_match")

Find all players (from Player) who never appeared in any match (not in Player_match). Show their Player_Name.

In [0]:
%sql
select distinct(p.player_id),p.player_name  from player p left join player_match pm on p.player_id = pm.player_id 

### fidn the player who played for more then one Team in desc order



In [0]:
%sql
select Player_Name, count(distinct(Player_Team) ) total_Team from player_match
group by Player_Name
order by total_team desc

###
 find the top 3 top  batsman from each team

In [0]:
%sql
select Striker,Team_Batting,total_Run,rank from(select Team_Batting,Striker,sum(Runs_Scored)as total_Run,dense_rank() over(partition by Team_Batting order by sum(Runs_Scored) desc) as rank  from ball_by_ball
group by Team_Batting,striker
order by Team_Batting,total_Run desc) s
where rank<3

find the average bowler and good bowler based on his extra performance 

In [0]:
%sql
select bowler, sum(Noballs), sum(Bowler_Extras), sum(Wides), sum(Penalty), p.player_name,
case 
  when sum(Noballs)+sum(Bowler_Extras)+sum(Wides)+sum(Penalty) > 60 then 'below average bowler'
  when sum(Noballs)+sum(Bowler_Extras)+sum(Wides)+sum(Penalty) between 41 and 60 then 'average bowler'
  when sum(Noballs)+sum(Bowler_Extras)+sum(Wides)+sum(Penalty) between 21 and 40 then 'good bowler'
  else 'excellent bowler'
end as bowler_profile
from ball_by_ball b 
join player p on b.Bowler = p.player_id
group by Bowler, p.player_name
order by sum(Noballs)+sum(Bowler_Extras)+sum(Wides)+sum(Penalty) desc

maximum win margin by team

In [0]:
%sql
--method1
SELECT match_winner,
       SUM(CASE WHEN Win_Margin = 'NULL' OR Win_Margin IS NULL THEN 0
                ELSE CAST(Win_Margin AS BIGINT)
           END) AS total_win_margin
FROM match
GROUP BY match_winner
order by total_win_margin desc ;

--method 2
SELECT match_winner,
       SUM(COALESCE(TRY_CAST(Win_Margin AS BIGINT), 0)) AS total_win_margin
FROM match
GROUP BY match_winner;
